#### Given a clickstream of user activity data, find the relevant user session for each click event.

- Session definition:
- session expires after inactivity of 30mins, because of inactivity no clickstream will be generated
- Session remains active for a total of 2 hours

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, unix_timestamp, lag, when, lit, concat, sum, monotonically_increasing_id
from pyspark.sql.window import Window

In [0]:
spark = SparkSession.builder \
    .appName("ClickStreamSession") \
    .getOrCreate()

In [0]:
schema = "click_time STRING, user_id STRING"

In [0]:
data = [
    ("2018-01-01 11:00:00", "u1"),
    ("2018-01-01 12:00:00", "u1"),
    ("2018-01-01 13:00:00", "u1"),
    ("2018-01-01 13:00:00", "u1"),
    ("2018-01-01 14:00:00", "u1"),
    ("2018-01-01 15:00:00", "u1"),
    ("2018-01-01 11:00:00", "u2"),
    ("2018-01-02 11:00:00", "u2")
]

In [0]:
clickstream_df = spark.createDataFrame(data, schema=schema)

In [0]:
# Convert click_time to Unix timestamp for easier calculations
clickstream_df = clickstream_df.withColumn("click_timestamp", unix_timestamp("click_time"))
session_window = Window.partitionBy("user_id").orderBy("click_timestamp")

In [0]:
clickstream_df = clickstream_df.withColumn("prev_click_timestamp", lag("click_timestamp", 1).over(session_window))

In [0]:
# Difference between click time and dividing that with 60
clickstream_df = clickstream_df.withColumn("timestamp_diff", (col("click_timestamp")-col("prev_click_timestamp"))/60)


In [0]:
# Updating null with 0
clickstream_df = clickstream_df.withColumn("timestamp_diff", when(col("timestamp_diff").isNull(), 0).otherwise(col("timestamp_diff")))


In [0]:
# Check for new session
clickstream_df = clickstream_df.withColumn("session_new", when(col("timestamp_diff") > 30, 1).otherwise(0))


In [0]:
# New session names
clickstream_df = clickstream_df.withColumn("session_new_name", concat(col("user_id"), lit("--S"), sum(col("session_new")).over(session_window)))


In [0]:
clickstream_df.show()


+-------------------+-------+---------------+--------------------+--------------+-----------+----------------+
|         click_time|user_id|click_timestamp|prev_click_timestamp|timestamp_diff|session_new|session_new_name|
+-------------------+-------+---------------+--------------------+--------------+-----------+----------------+
|2018-01-01 11:00:00|     u1|     1514804400|                NULL|           0.0|          0|          u1--S0|
|2018-01-01 12:00:00|     u1|     1514808000|          1514804400|          60.0|          1|          u1--S1|
|2018-01-01 13:00:00|     u1|     1514811600|          1514808000|          60.0|          1|          u1--S2|
|2018-01-01 13:00:00|     u1|     1514811600|          1514811600|           0.0|          0|          u1--S2|
|2018-01-01 14:00:00|     u1|     1514815200|          1514811600|          60.0|          1|          u1--S3|
|2018-01-01 15:00:00|     u1|     1514818800|          1514815200|          60.0|          1|          u1--S4|
|